<a href="https://colab.research.google.com/github/yogendra785/suspicious-login-detection-pytorch/blob/main/Suspicious_Login_Behavior_Detection.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd
import numpy as np
import random


np.random.seed(42)
random.seed(42)

def generate_login_data(num_samples=5000):
    data = []

    for _ in range(num_samples):

        is_suspicious = 1 if random.random() < 0.2 else 0

        if is_suspicious == 0:
            # NORMAL LOGIN BEHAVIOR
            distance_km = random.uniform(0, 50) # Same city, typical commute
            time_since_last_min = random.uniform(120, 2880) # A few hours to a couple of days
            failed_attempts = random.choice([0, 0, 0, 1]) # Might mistype a password once
            device_type = 0 # 0 = Known device

        else:
            # SUSPICIOUS LOGIN BEHAVIOR (e.g., "Impossible Travel" or Brute Force)
            scenario = random.choice(['impossible_travel', 'brute_force'])

            if scenario == 'impossible_travel':
                distance_km = random.uniform(1000, 10000) # Halfway across the world
                time_since_last_min = random.uniform(1, 15) # But only 5 minutes since last login!
                failed_attempts = random.choice([0, 1])
                device_type = random.choice([0, 1])

            elif scenario == 'brute_force':
                distance_km = random.uniform(0, 500)
                time_since_last_min = random.uniform(1, 60)
                failed_attempts = random.randint(5, 15) # Hammering the login button
                device_type = 1 # 1 = Unknown device

        data.append([distance_km, time_since_last_min, failed_attempts, device_type, is_suspicious])

    columns = ['distance_from_last_login_km', 'time_since_last_login_min',
               'failed_attempts_24h', 'is_unknown_device', 'is_suspicious']

    return pd.DataFrame(data, columns=columns)


df = generate_login_data(5000)

# Round the float columns for a cleaner look
df['distance_from_last_login_km'] = df['distance_from_last_login_km'].round(2)
df['time_since_last_login_min'] = df['time_since_last_login_min'].round(2)


print("🚀 Dataset Snapshot:")
display(df.head())

print("\n📊 Class Distribution (0 = Normal, 1 = Suspicious):")
print(df['is_suspicious'].value_counts())

🚀 Dataset Snapshot:


,distance_from_last_login_km,time_since_last_login_min,failed_attempts_24h,is_unknown_device,is_suspicious
0,1.25,879.08,0,0,0
1,7090.30,13.49,0,1,1
2,2967.74,8.07,0,0,1
3,35.07,1277.87,1,0,0
4,40.47,137.94,0,0,0



📊 Class Distribution (0 = Normal, 1 = Suspicious):
is_suspicious
0    3938
1    1062
Name: count, dtype: int64


In [2]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5000 entries, 0 to 4999
Data columns (total 5 columns):
 #   Column                       Non-Null Count  Dtype  
---  ------                       --------------  -----  
 0   distance_from_last_login_km  5000 non-null   float64
 1   time_since_last_login_min    5000 non-null   float64
 2   failed_attempts_24h          5000 non-null   int64  
 3   is_unknown_device            5000 non-null   int64  
 4   is_suspicious                5000 non-null   int64  
dtypes: float64(2), int64(3)
memory usage: 195.4 KB


In [3]:
import torch
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

X=df.drop('is_suspicious',axis=1).values
y=df['is_suspicious'].values

X_train, X_test, y_train,y_test = train_test_split(X,y,test_size=0.2,random_state=42)

scaler = StandardScaler()
X_train_scaled=scaler.fit_transform(X_train)
X_test_scaled=scaler.fit_transform(X_test)

#Defining our custom pytorch datasets

class LoginDataset(Dataset):
  def __init__(self,features,labels):
    self.features=torch.FloatTensor(features)
    self.labels=torch.FloatTensor(labels).unsqueeze(1)

  def __len__(self):
    return len(self.labels)

  def __getitem__(self,idx):
    return self.features[idx],self.labels[idx]

train_dataset=LoginDataset(X_train_scaled,y_train)
test_dataset=LoginDataset(X_test_scaled,y_test)

BATCH_SIZE=64
train_loader = DataLoader(train_dataset,batch_size=BATCH_SIZE,shuffle=True)
test_loader= DataLoader(test_dataset, batch_size=BATCH_SIZE,shuffle=False)


print("✅ Data successfully converted to PyTorch Tensors!")
print(f"Number of training batches: {len(train_loader)}")

# Let's peek at exactly what one batch looks like
sample_features, sample_labels = next(iter(train_loader))
print(f"\nShape of one batch of features: {sample_features.shape} -> (Batch Size, Number of Features)")
print(f"Shape of one batch of labels: {sample_labels.shape} -> (Batch Size, Label)")



✅ Data successfully converted to PyTorch Tensors!
Number of training batches: 63

Shape of one batch of features: torch.Size([64, 4]) -> (Batch Size, Number of Features)
Shape of one batch of labels: torch.Size([64, 1]) -> (Batch Size, Label)


In [4]:
import torch
import torch.nn as nn

class LoginBehaviorModel(nn.Module):
  def __init__(self):
    super(LoginBehaviorModel,self).__init__()

    #using nn.Sequential to cleanly stack our layers
    self.network = nn.Sequential(
        #input layer ->hidden layer 1
        nn.Linear(in_features=4,out_features=16),
        nn.ReLU(),

        #hidden layer 1 ->hidden layer 2
        nn.Linear(in_features=16, out_features=8),
        nn.ReLU(),

        #hidden layer 2 ->output layer
        nn.Linear(in_features=8,out_features=1),
        nn.Sigmoid()

    )
  def forward(self,x):
    return self.network(x)
#creating instance of the model
model = LoginBehaviorModel()
print("Neural Network Architecture Defined!")
print(model)

Neural Network Architecture Defined!
LoginBehaviorModel(
  (network): Sequential(
    (0): Linear(in_features=4, out_features=16, bias=True)
    (1): ReLU()
    (2): Linear(in_features=16, out_features=8, bias=True)
    (3): ReLU()
    (4): Linear(in_features=8, out_features=1, bias=True)
    (5): Sigmoid()
  )
)


In [5]:
#passing the sample batch created in phase 2 through untrained model
untrained_predictions=model(sample_features)
print(f"\nShape of predictions: {untrained_predictions.shape} -> (Batch Size, Probability Score)")
print(f"First 5 untrained predictions:\n{untrained_predictions[:5].detach().numpy()}")



Shape of predictions: torch.Size([64, 1]) -> (Batch Size, Probability Score)
First 5 untrained predictions:
[[0.43690595]
 [0.45971736]
 [0.45996615]
 [0.4582633 ]
 [0.40679014]]


In [11]:
import torch.optim as optim
#defining a loss function
# BCELoss = Binary Cross Entropy (Standard for Yes/No, 1/0 predictions)
criterion = nn.BCELoss()
optimizer = optim.Adam(model.parameters(),lr=0.01)

#training loop
EPOCHS=20
print("----Started Training----\n")

for epoch in range(EPOCHS):
  model.train()
  total_loss=0
  correct_predictions=0
  total_samples=0

  for batch_features, batch_labels in train_loader:
    optimizer.zero_grad()

    predictions=model(batch_features)

    loss=criterion(predictions, batch_labels)

    loss.backward()

    optimizer.step()

    total_loss +=loss.item()

    binary_preds=(predictions > 0.5).float()
    correct_predictions += (binary_preds == batch_labels).sum().item()
    total_samples += batch_labels.size(0)

  avg_loss = total_loss / len(train_loader)
  accuracy = (correct_predictions / total_samples) *100
  if (epoch+1)%5==0 or epoch == 0:
    print(f"Epoch [{epoch+1:02d}/{EPOCHS}] | Loss: {avg_loss:.4f} | Training Accuracy: {accuracy:.2f}%")
print("\n Training completed!")





----Started Training----

Epoch [01/20] | Loss: 0.1616 | Training Accuracy: 97.08%
Epoch [05/20] | Loss: 0.0015 | Training Accuracy: 99.98%
Epoch [10/20] | Loss: 0.0011 | Training Accuracy: 99.98%
Epoch [15/20] | Loss: 0.0008 | Training Accuracy: 100.00%
Epoch [20/20] | Loss: 0.0006 | Training Accuracy: 100.00%

 Training completed!


In [12]:
#evaluating the model on unseen test data

model.eval()
test_correct=0
test_total=0

with torch.no_grad():
  for batch_features, batch_labels in test_loader:
    predictions = model(batch_features)
    binary_preds = (predictions > 0.5).float()

    test_correct += (binary_preds == batch_labels).sum().item()
    test_total += batch_labels.size(0)
test_accuracy = (test_correct / test_total)*100
print(f"Final test set accuracy: {test_accuracy:.2f}%\\n")

#Explainable Ai Inference Engine

def predict_and_explain(distance, time_min, fails, unknown_device):
   """
    Takes raw user login data, scales it, feeds it to the PyTorch model,
    and outputs an explainable security decision.
    """
   raw_data = [distance,time_min,fails, unknown_device]
   scaled_data = scaler.transform([raw_data])
   tensor_data = torch.FloatTensor(scaled_data)

   with torch.no_grad():
    score = model(tensor_data).item()

   is_suspicious = score > 0.5

   print("--------------------------------------------------")
   print(f"🔍 ANALYZING LOGIN: Distance={distance}km, Time={time_min}min, Fails={fails}, UnknownDev={unknown_device}")
   print(f"⚠️ Threat Score: {score*100:.2f}%")

   if is_suspicious:
    print("VERDICT: BLOCKED")
    print("Model Reasoning:")

    if distance > 500 and time_min <60:
      print("   ↳ 'Impossible Travel' detected (High distance in physically impossible time frame).")

    if fails >=3:
      print("  ↳ Brute force indicator: Multiple failed attempts.")
    if unknown_device == 1:
      print("   ↳ Session initiated from an unrecognized device.")
   else:
     print("✅ VERDICT: APPROVED")
     print("🧠 Model Reasoning: Metrics align with established normal behavior patterns.")
   print("--------------------------------------------------\n")

# Testing

# Scenario A: A normal user logging in from their home office the next morning
predict_and_explain(distance=5, time_min=720, fails=0, unknown_device=0)

# Scenario B: A hacker in another country buying a stolen password
predict_and_explain(distance=6000, time_min=10, fails=6, unknown_device=1)











Final test set accuracy: 100.00%\n
--------------------------------------------------
🔍 ANALYZING LOGIN: Distance=5km, Time=720min, Fails=0, UnknownDev=0
⚠️ Threat Score: 0.00%
✅ VERDICT: APPROVED
🧠 Model Reasoning: Metrics align with established normal behavior patterns.
--------------------------------------------------

--------------------------------------------------
🔍 ANALYZING LOGIN: Distance=6000km, Time=10min, Fails=6, UnknownDev=1
⚠️ Threat Score: 100.00%
VERDICT: BLOCKED
Model Reasoning:
   ↳ 'Impossible Travel' detected (High distance in physically impossible time frame).
  ↳ Brute force indicator: Multiple failed attempts.
   ↳ Session initiated from an unrecognized device.
--------------------------------------------------



In [13]:
import torch
import pickle

# 1. Save the PyTorch Model weights (The Brain)
# We save the 'state_dict', which is a dictionary containing all the learned weights and biases
torch.save(model.state_dict(), 'login_model_weights.pth')
print("✅ PyTorch Model weights saved as 'login_model_weights.pth'")

# 2. Save the StandardScaler (The Translator)
# We use pickle to save the exact scaling parameters learned from our training data
with open('scaler.pkl', 'wb') as f:
    pickle.dump(scaler, f)
print("✅ StandardScaler saved as 'scaler.pkl'")

✅ PyTorch Model weights saved as 'login_model_weights.pth'
✅ StandardScaler saved as 'scaler.pkl'


In [10]:
import torch.nn as nn

class LoginBehaviorModel(nn.Module):
    def __init__(self):
        super(LoginBehaviorModel, self).__init__()

        self.network = nn.Sequential(
            # Input Layer -> Hidden Layer 1
            nn.Linear(in_features=4, out_features=16),
            nn.ReLU(),
            nn.Dropout(p=0.2), #20% of neurons drop out

            # Hidden Layer 1 -> Hidden Layer 2
            nn.Linear(in_features=16, out_features=8),
            nn.ReLU(),
            nn.Dropout(p=0.2), #Another 20% drop out

            # Hidden Layer 2 -> Output Layer
            nn.Linear(in_features=8, out_features=1),
            nn.Sigmoid()
        )

    def forward(self, x):
        return self.network(x)

# Instantiate the upgraded model
model = LoginBehaviorModel()

print("✅ Upgraded Neural Network with Dropout Defined!")
print(model)

✅ Upgraded Neural Network with Dropout Defined!
LoginBehaviorModel(
  (network): Sequential(
    (0): Linear(in_features=4, out_features=16, bias=True)
    (1): ReLU()
    (2): Dropout(p=0.2, inplace=False)
    (3): Linear(in_features=16, out_features=8, bias=True)
    (4): ReLU()
    (5): Dropout(p=0.2, inplace=False)
    (6): Linear(in_features=8, out_features=1, bias=True)
    (7): Sigmoid()
  )
)
